# RESP with Multiple Spatial Rotations

The example mimics what was done with R.E.D., where each conformation was rotated in Cartesian space to supply a different orientation in computing the ESP. By providing different orientations, the derived partial atomic charges should become reproducible to a higher decimal precision.

"Every molecular orientation can potentially yield different charge values. Consequently, a structure has to be re-oriented to lead to reproducible RESP or ESP charges, and the method used for reorientation needs to be reported." [1]

Specs:
- 1 conformation per molecule species (i.e., with and without EP)
- 4 spatial orientations per molecule species
- One-stage RESP is performed
    - Stage 1: stage_1.ini

**References**
1. Dupradeau, F.-Y.; Pigache, A.; Zaffran, T.; Savineau, C.; Lelong, R.; Grivel, N.; Lelong, D.; Rosanski, W. & Cieplak, P. The R.E.D. tools: advances in RESP and ESP charge derivation and force field library building Phys Chem Chem Phys, 2010, 12, 7821-7839

<br>

**IMPORTANT**:
1. The two input species (i.e., with and without the EP) need to be **identical**, apart from the extra points themselves.
    - *Why*: The QM MEP target data is computed exclusively using the standard molecular structures (without EPs). Because the underlying geometries and spatial orientations are identical, these same MEP grids can be directly reused for the fitting that includes EPs. Consequently, this eliminates the need to perform redundant, computationally expensive QM calculations on structures containing EPs.
2. High values for `num_variants` (see `generate_rotated_molecules` below) **may produce duplicate** molecular **orientations**.
    - *Why*: This occurs because the fixed rotation steps will eventually hit $360^\circ$ and wrap around to match earlier variants. However, this is needed so that structures with and without the EPs overlap, thereby reducing the number of QM MEPs to be computed.

In [1]:
import os
from pathlib import Path
import sys

import numpy as np
from scipy.spatial.transform import Rotation as R

In [2]:
import psi4

In [ ]:
from resp_ep import driver
from resp_ep import extras

## Create rotational variants of the input

- Probably not needed since you already have teh *.dat files from a previous RESP calculstions.

In [4]:
def rotate_xyz(xyz: list, x_rotation: float, y_rotation: float, z_rotation: float) -> list:
    """ Rotates 3D Cartesian coordinates sequentially around the X, Y, and Z axes.

        Args:
            xyz: A list of [x, y, z] coordinates to rotate.
            x_rotation: Rotation angle around the X-axis in degrees.
            y_rotation: Rotation angle around the Y-axis in degrees.
            z_rotation: Rotation angle around the Z-axis in degrees.

        Returns:
            A list of the newly rotated [x, y, z] coordinates.
    """
    rotation = R.from_euler('xyz', [x_rotation, y_rotation, z_rotation], degrees=True)
    return rotation.apply(xyz).tolist()


def generate_rotamers(basename, num_variants=4, step=72):
    """ Generates unique spatial variants of a molecule by cycling isolated axis rotations.

    Args:
        basename: The prefix name of the input/output .xyz files.
        num_variants: Total number of rotated conformers to generate.
        step: The increment angle in degrees used to step through rotations.

    Warning:
        High values for num_variants in combo with the step size may result in 
        duplicate spatial orientations. Since the rotations are cyclic, any total axis 
        rotation that reaches or exceeds 360 degrees will replicate a previously 
        generated conformer.
    """
    input_file = f'{basename}.xyz'
    atoms, xyz_coords = extras.parse_xyz(input_file)

    alphabet = "abcdefghijklmnopqrstuvwxyz"
    suffixes = alphabet[0 : num_variants]  # e.g., "a, b, c, d" if num_variants=4

    for i, suffix in enumerate(suffixes):
        output_file = f'{basename}_{suffix}.xyz'

        # Maximize differences: Isolate rotations to distinct axes per file
        # conf. a: [0, 0, 0], conf. b: [72, 0, 0], conf c: [0, 144, 0], conf d: [216, 0, 0, ]
        angle = step * i
        rotations = [0, 0, 0]
        rotations[i % 3] = angle

        new_coords = [rotate_xyz(xyz, *rotations) for xyz in xyz_coords]

        extras.write_xyz(elements=atoms, xyz=new_coords, output_filepath=output_file)
        print(f"Generated: {output_file} with rotations {rotations}")

In [5]:
basename = 'acetic_acid'
generate_rotamers(basename, num_variants=4)

basename = 'acetic_acid_x'
generate_rotamers(basename, num_variants=4)

Generated: acetic_acid_a.xyz with rotations [0, 0, 0]
Generated: acetic_acid_b.xyz with rotations [0, 72, 0]
Generated: acetic_acid_c.xyz with rotations [0, 0, 144]
Generated: acetic_acid_d.xyz with rotations [216, 0, 0]
Generated: acetic_acid_x_a.xyz with rotations [0, 0, 0]
Generated: acetic_acid_x_b.xyz with rotations [0, 72, 0]
Generated: acetic_acid_x_c.xyz with rotations [0, 0, 144]
Generated: acetic_acid_x_d.xyz with rotations [216, 0, 0]


## Perform a Stage 1 Fitting
- No EP centers present.

In [6]:
stage_1 = f'''\
[molecules]
input_files: acetic_acid_a.xyz,
             acetic_acid_b.xyz,
             acetic_acid_c.xyz,
             acetic_acid_d.xyz
             
[vdw.surface.options]
vdw_scale_factors : 1.4, 1.6, 1.8, 2.0
vdw_point_density : 1.0
esp               : None
grid              : None

vdw_radii         : None

[hyperbolic.restraint.options]
weight            : 1.0, 1.0, 1.0, 1.0
restraint         : True
resp_a            : 0.0005
resp_b            : 0.1
toler             : 1e-5
max_it            : 50 
ihfree            : True

[constraints]
constraint_charge : None
equivalent_groups : None

[qm.options]
method_esp : b3lyp
basis_esp  : assign H 6-311++G**,
             assign C 6-311++G**,
             assign O 6-311++G**

formal_charge : 0
multiplicity  : 1
'''

with open('stage_1.ini', 'w') as file:
    file.write(stage_1)

In [7]:
charges_stage_1 = driver.resp(input_ini='stage_1.ini')
print()
print(f'ESP:\n{charges_stage_1[0]}\n')
print(f'RESP Stage 1:\n{charges_stage_1[1]}')


Determining Partial Atomic Charges

INPUT COORDS, ANGSTROMS
 [[ 1.45051389 -0.06628932  0.        ]
 [ 1.75521613 -0.62865986 -0.87500146]
 [ 1.75521613 -0.62865986  0.87500146]
 [ 1.92173244  0.90485897  0.        ]
 [-0.04233122  0.09849378  0.        ]
 [-0.67064817 -1.07620915  0.        ]
 [-1.60837259 -0.91016601  0.        ]
 [-0.62675864  1.1316051   0.        ]]
1.910118
1.674686
1.674686
1.674686
1.910118
1.714534
1.674686
1.714534
PSI4:  "['assign H 6-311++G**', 'assign C 6-311++G**', 'assign O 6-311++G**']"
Points: 773; Coord: 8
INPUT COORDS, ANGSTROMS
 [[ 0.44823344 -0.06628932 -1.37952069]
 [-0.28978423 -0.62865986 -1.93970006]
 [ 1.37456745 -0.62865986 -1.39891942]
 [ 0.59384798  0.90485897 -1.82767616]
 [-0.01308107  0.09849378  0.04025938]
 [-0.20724168 -1.07620915  0.63782431]
 [-0.49701446 -0.91016601  1.52965323]
 [-0.19367907  1.1316051   0.59608289]]
1.910118
1.674686
1.674686
1.674686
1.910118
1.714534
1.674686
1.714534
PSI4:  "['assign H 6-311++G**', 'assign C 

## Extra Points
- 4 EP centers present, 2 on each oxygen atoms.

In [8]:
stage_1_x = f'''\
[molecules]
input_files: acetic_acid_x_a.xyz,
             acetic_acid_x_b.xyz,
             acetic_acid_x_c.xyz,
             acetic_acid_x_d.xyz

[vdw.surface.options]
vdw_radii         : X  = 0.0
vdw_scale_factors : 1.4, 1.6, 1.8, 2.0
vdw_point_density : 1.0

esp               : acetic_acid_a_esp.dat,
                    acetic_acid_b_esp.dat,
                    acetic_acid_c_esp.dat,
                    acetic_acid_d_esp.dat

grid              : acetic_acid_a_grid.dat,
                    acetic_acid_b_grid.dat,
                    acetic_acid_c_grid.dat,
                    acetic_acid_d_grid.dat

[hyperbolic.restraint.options]
weight            : 1.0, 1.0, 1.0, 1.0
restraint         : True
resp_a            : 0.0005
resp_b            : 0.1
ihfree            : True
max_it            : 50 
toler             : 1e-5

[constraints]
constraint_charge : None
equivalent_groups : None

[qm.options]
method_esp : None
basis_esp  : None

formal_charge : 0
multiplicity  : 1
'''

with open('stage_1_x.ini', 'w') as file:
    file.write(stage_1_x)

In [9]:
charges_stage_1 = driver.resp(input_ini='stage_1_x.ini')
print()
print(f'ESP:\n{charges_stage_1[0]}\n')
print(f'RESP Stage 1:\n{charges_stage_1[1]}')


Determining Partial Atomic Charges

INPUT COORDS, ANGSTROMS
 [[ 1.45051389 -0.06628932  0.        ]
 [ 1.75521613 -0.62865986 -0.87500146]
 [ 1.75521613 -0.62865986  0.87500146]
 [ 1.92173244  0.90485897  0.        ]
 [-0.04233122  0.09849378  0.        ]
 [-0.67064817 -1.07620915  0.        ]
 [-1.60837259 -0.91016601  0.        ]
 [-0.62675864  1.1316051   0.        ]
 [-0.58238598 -1.25790702  0.28582454]
 [-0.58238598 -1.25790702 -0.28582454]
 [-0.44910278  1.43316504  0.        ]
 [-0.97674514  1.13467962  0.        ]]
1.910118
1.674686
1.674686
1.674686
1.910118
1.714534
1.674686
1.714534
Points: 773; Coord: 12
INPUT COORDS, ANGSTROMS
 [[ 0.44823344 -0.06628932 -1.37952069]
 [-0.28978423 -0.62865986 -1.93970006]
 [ 1.37456745 -0.62865986 -1.39891942]
 [ 0.59384798  0.90485897 -1.82767616]
 [-0.01308107  0.09849378  0.04025938]
 [-0.20724168 -1.07620915  0.63782431]
 [-0.49701446 -0.91016601  1.52965323]
 [-0.19367907  1.1316051   0.59608289]
 [ 0.09186813 -1.25790702  0.64220662